# 08. 反馈标签精细分布 & 动作词映射（构造正负样本 & 兴趣演化必备）

> 对应 `analysis_plan.md` §2.8。**§2.5 已算了每个 label 的整体正样本率**；本 notebook 补充 §2.5 未覆盖的：

## 目的（§2.5 未涵盖的部分）

1. **多值 label 精确分布**：`ec_cvr_label_list` / `ec_colossus_rs_is_click/cart/buy` / `video_*_list` 各取值的具体计数（0/1/2+ 各占多少）。
2. **计数类 label 分位数**：`live_hist_valid_play_cnt_list` / `like_cnt / comment_cnt / play_duration` 等 count 型标签的**具体值分布**（P50/P90/P99），决定「多次浏览 vs 单次浏览」的阈值 —— **兴趣演化任务的关键**。
3. **字符串枚举分布**：`click_type` / `click_industry` / `live_hist_author_category_type_list` / `live_hist_author_type_list` 的 top-K 类别。
4. **Live 直播类型布尔字段**：`is_interactive_mp_live / is_building_live / is_local_life_live / is_detect_game_live / is_recruit_live` 各自的正样本率。
5. **⭐ 动作词映射表**：把原始 label 映射到评估侧中文动作词（如 `play_done=1` → `[视频-长播]`，`is_buy=1` → `[商品-购买]`，`follow=1` → `[直播-关注]`）；这是 SFT prompt 里 timeline 的动作词与评估侧对齐的**核心表**。

## 输入 / 输出

**输入**：`OneReason_UserProfile/*.parquet`（500K 用户，抽样 5% 做字符串枚举 + 分位数分析）

**输出**（`analysis/outputs/labels/`）：
- `numeric_label_dist.csv` —— 数值 label 的取值精确分布
- `count_label_percentiles.csv` —— count 型 label 的 P50/P90/P99
- `str_enum_dist_{field}.csv` —— 字符串枚举字段的 top-K
- `live_type_flags.csv` —— live 类型布尔字段正样本率
- `action_phrase_mapping.csv` —— **⭐ 动作词映射表**（label → 中文动作短语）
- `_cache.pkl`

## 关键设计

**抽样 5% 用户**（25K）做字符串枚举和 count-type 分位数（数据量已经充足），避免 3500 万条 string 全量扫描导致的 to_pylist 慢问题。**布尔类字段**因为存储简单，走全量。

In [1]:
# ============ 配置 & imports ============
import os, pickle, time
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Songti SC', 'STHeiti', 'Hiragino Sans GB', 'PingFang SC', 'Heiti SC', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path('/Users/cjs/Desktop/MY/explore_llm_rec')
USER_DIR     = PROJECT_ROOT / 'dataset_rec_all' / 'data' / 'OneReason_UserProfile'
OUT_DIR      = PROJECT_ROOT / 'analysis' / 'outputs' / 'labels'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH   = OUT_DIR / '_cache.pkl'

USE_CACHE = True
SAMPLE_FRAC_STR   = 0.05    # 字符串枚举抽 5%
SAMPLE_FRAC_COUNT = 0.10    # count 类分位数抽 10%（数值列快）

# ---- 字段分组 ----

# 数值 label：期望取值极小（0/1 或 0/1/2+），做完整 value_counts
NUM_LABELS = [
    'video_like_list', 'video_comment_list', 'video_forward_list',
    'video_collect_list', 'video_neg_feedback_list', 'video_play_done_list',
    'video_history_like_list', 'video_history_comment_list', 'video_history_forward_list',
    'video_history_collect_list', 'video_history_neg_feedback_list', 'video_history_play_done_list',
    'ec_cvr_label_list',
    'ec_colossus_rs_is_click_list', 'ec_colossus_rs_is_cart_list', 'ec_colossus_rs_is_buy_list',
    'ec_colossus_rs_lagv1_list', 'ec_colossus_rs_lagv2_list',
    'ec_trunc_clk_lag', 'ec_trunc_buy_lag',
]

# count-type label：整数取值范围广，做 P50/P90/P99
COUNT_LABELS = [
    'live_hist_show_cnt_list', 'live_hist_play_cnt_list', 'live_hist_valid_play_cnt_list',
    'live_hist_play_duration_list', 'live_hist_valid_play_duration_list',
    'live_hist_like_cnt_list', 'live_hist_comment_cnt_list',
    'live_hist_reduce_similar_cnt_list', 'live_hist_report_live_cnt_list',
    'live_hist_follow_author_cnt_list',
    'video_watch_time_list', 'video_duration_list',
    'video_history_watch_time_list', 'video_history_duration_list',
]

# 字符串枚举：value_counts top-K
STR_ENUMS = [
    'live_hist_author_category_type_list',
    'live_hist_author_type_list',
    'outer_loop_history_action_pid_list_click_type',
    'outer_loop_history_action_pid_list_click_industry',
]

# Live 直播类型布尔标签
LIVE_FLAGS = [
    'live_hist_is_interactive_mp_live_list',
    'live_hist_is_building_live_list',
    'live_hist_is_local_life_live_list',
    'live_hist_is_detect_game_live_list',
    'live_hist_is_recruit_live_list',
]

print('User dir  :', USER_DIR)
print('Output dir:', OUT_DIR)
parquet_files = sorted(USER_DIR.glob('part-*.parquet'))
print(f'#parquet  : {len(parquet_files)}')
print(f'#num_labels={len(NUM_LABELS)}  #count_labels={len(COUNT_LABELS)}  #str_enums={len(STR_ENUMS)}  #live_flags={len(LIVE_FLAGS)}')

User dir  : /Users/cjs/Desktop/MY/explore_llm_rec/dataset_rec_all/data/OneReason_UserProfile
Output dir: /Users/cjs/Desktop/MY/explore_llm_rec/analysis/outputs/labels
#parquet  : 10
#num_labels=20  #count_labels=14  #str_enums=4  #live_flags=5


## 单遍扫描：一次收集数值/count/字符串/布尔字段

- **数值字段**：flat 全量 `value_counts`（因为 int64 快）
- **count 字段**：抽样 10% 用户 → flat 池，最后计算 P50/P90/P99
- **字符串字段**：抽样 5% 用户 → arrow `value_counts` 累积（避免 to_pylist）
- **布尔字段**：全量 `value_counts`

In [2]:
def _num_valcount(chunked_col):
    """list<int/float> chunked -> dict {val: count}. 只取 <= 20 个 distinct（防错爆）"""
    if chunked_col is None: return {}
    flat = chunked_col.combine_chunks().values
    if pa.types.is_floating(flat.type):
        arr = np.asarray(flat, dtype=np.float64)
        # 视为 float 后按整数 bin（0.0/1.0 也归为 0/1）
        arr = np.rint(arr).astype(np.int64)
    else:
        arr = np.asarray(flat, dtype=np.int64)
    if arr.size == 0: return {}
    uniq, cnt = np.unique(arr, return_counts=True)
    if uniq.size > 30:
        # 太多 distinct，说明不是 label 而是 count 类，只保留 top-20
        order = np.argsort(cnt)[::-1][:20]
        return {int(uniq[i]): int(cnt[i]) for i in order}
    return {int(u): int(c) for u, c in zip(uniq, cnt)}


def _count_flat_sample(chunked_col, sample_rate=1.0, cap=200_000):
    """取抽样后的 flat np array（用于分位数）"""
    if chunked_col is None: return np.array([], dtype=np.int64)
    flat = chunked_col.combine_chunks().values
    if flat is None or len(flat) == 0: return np.array([], dtype=np.int64)
    if pa.types.is_floating(flat.type):
        arr = np.asarray(flat, dtype=np.float64)
    else:
        arr = np.asarray(flat, dtype=np.int64)
    # 抽样：总量 * sample_rate
    if sample_rate < 1.0 and arr.size > cap:
        rng = np.random.default_rng(2026)
        idx = rng.choice(arr.size, size=min(cap, int(arr.size * sample_rate)), replace=False)
        arr = arr[idx]
    elif arr.size > cap:
        rng = np.random.default_rng(2026)
        idx = rng.choice(arr.size, size=cap, replace=False)
        arr = arr[idx]
    return arr


def _str_valcount(chunked_col, cap=100_000):
    """list<string> chunked -> Counter (arrow value_counts)."""
    if chunked_col is None: return {}
    flat = chunked_col.combine_chunks().values
    if flat is None or len(flat) == 0: return {}
    # 抽样后再算，避免全量太慢
    n = len(flat)
    if n > cap:
        idx = np.random.default_rng(2026).choice(n, size=cap, replace=False)
        flat = flat.take(pa.array(idx))
    vc = pc.value_counts(flat)
    out = {}
    for r in vc.to_pylist():
        v = r['values']; c = r['counts']
        if v is None or v == '': continue
        out[str(v)] = out.get(str(v), 0) + int(c)
    return out


def scan_labels(files):
    num_dist    = defaultdict(lambda: defaultdict(int))    # field -> {val: count}
    count_pool  = defaultdict(list)                        # field -> list of np.array
    str_dist    = defaultdict(lambda: defaultdict(int))    # field -> {val: count}
    flag_dist   = defaultdict(lambda: defaultdict(int))    # field -> {0: n, 1: n}

    read_cols = list(set(NUM_LABELS + COUNT_LABELS + STR_ENUMS + LIVE_FLAGS))
    t0 = time.time()
    for i, fp in enumerate(files):
        if fp.stat().st_size == 0: continue
        # 只读需要的列
        avail = pq.ParquetFile(fp).schema_arrow.names
        cols = [c for c in read_cols if c in avail]
        tbl = pq.read_table(fp, columns=cols)
        # 抽样一份用于 str + count（同一份切片，节省内存）
        n = tbl.num_rows
        take_str   = max(1, int(n * SAMPLE_FRAC_STR))
        take_count = max(1, int(n * SAMPLE_FRAC_COUNT))
        rng = np.random.default_rng(2026 + i)
        idx_str   = np.sort(rng.choice(n, size=take_str, replace=False))
        idx_count = np.sort(rng.choice(n, size=take_count, replace=False))
        tbl_str   = tbl.take(pa.array(idx_str))
        tbl_count = tbl.take(pa.array(idx_count))

        # 数值 label：走全量
        for f in NUM_LABELS:
            if f not in tbl.column_names: continue
            for k, v in _num_valcount(tbl.column(f)).items():
                num_dist[f][k] += v

        # count-type：走 10% 抽样
        for f in COUNT_LABELS:
            if f not in tbl_count.column_names: continue
            arr = _count_flat_sample(tbl_count.column(f), sample_rate=1.0, cap=200_000)
            if arr.size > 0:
                count_pool[f].append(arr)

        # 字符串枚举：走 5% 抽样
        for f in STR_ENUMS:
            if f not in tbl_str.column_names: continue
            for k, c in _str_valcount(tbl_str.column(f), cap=100_000).items():
                str_dist[f][k] += c

        # live flags：走全量（int 快）
        for f in LIVE_FLAGS:
            if f not in tbl.column_names: continue
            for k, v in _num_valcount(tbl.column(f)).items():
                flag_dist[f][k] += v

        print(f'  parquet {i+1}/{len(files)} done  elapsed={time.time()-t0:.1f}s')

    count_pool_final = {f: (np.concatenate(v) if v else np.array([], dtype=np.int64))
                       for f, v in count_pool.items()}
    return {
        'num_dist':    {k: dict(v) for k, v in num_dist.items()},
        'count_pool':  count_pool_final,
        'str_dist':    {k: dict(v) for k, v in str_dist.items()},
        'flag_dist':   {k: dict(v) for k, v in flag_dist.items()},
    }


if USE_CACHE and CACHE_PATH.exists():
    print(f'[cache] loading {CACHE_PATH}')
    with open(CACHE_PATH, 'rb') as f:
        stats = pickle.load(f)
    print('[cache] loaded.')
else:
    stats = scan_labels(parquet_files)
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(stats, f)
    print(f'[cache] saved to {CACHE_PATH}')

print('\n=== fields collected ===')
print(f'  num_dist  : {len(stats["num_dist"])} fields')
print(f'  count_pool: {len(stats["count_pool"])} fields')
print(f'  str_dist  : {len(stats["str_dist"])} fields')
print(f'  flag_dist : {len(stats["flag_dist"])} fields')

  parquet 1/10 done  elapsed=4.4s
  parquet 2/10 done  elapsed=7.9s
  parquet 3/10 done  elapsed=11.5s
  parquet 4/10 done  elapsed=15.3s
  parquet 5/10 done  elapsed=19.6s
  parquet 6/10 done  elapsed=23.6s
  parquet 7/10 done  elapsed=27.2s
  parquet 8/10 done  elapsed=31.0s
  parquet 9/10 done  elapsed=35.1s
  parquet 10/10 done  elapsed=38.5s
[cache] saved to /Users/cjs/Desktop/MY/explore_llm_rec/analysis/outputs/labels/_cache.pkl

=== fields collected ===
  num_dist  : 20 fields
  count_pool: 14 fields
  str_dist  : 4 fields
  flag_dist : 5 fields


## §2.8.1 数值 Label 精确取值分布

In [3]:
num_rows = []
for f, dd in stats['num_dist'].items():
    total = sum(dd.values())
    if total == 0: continue
    row = {'field': f, 'total': total, 'distinct': len(dd)}
    for k in sorted(dd.keys())[:6]:
        row[f'val={k}'] = dd[k]
        row[f'val={k}_%'] = round(dd[k] / total * 100, 3)
    if len(dd) > 6:
        others = sum(v for k, v in dd.items() if k not in sorted(dd.keys())[:6])
        row['others'] = others
        row['others_%'] = round(others / total * 100, 3)
    num_rows.append(row)
num_df = pd.DataFrame(num_rows).fillna('')
num_df.to_csv(OUT_DIR / 'numeric_label_dist.csv', index=False)
print('saved:', OUT_DIR / 'numeric_label_dist.csv')
num_df

saved: /Users/cjs/Desktop/MY/explore_llm_rec/analysis/outputs/labels/numeric_label_dist.csv


,field,total,distinct,val=0,val=0_%,val=1,val=1_%,val=23,val=23_%,val=24,...,val=1350,val=1350_%,val=2,val=2_%,val=3,val=3_%,val=4,val=4_%,val=5,val=5_%
0,video_like_list,1656218,2,1587182.0,95.832,69036.0,4.168,,,,...,,,,,,,,,,
1,video_comment_list,1656218,2,1635374.0,98.741,20844.0,1.259,,,,...,,,,,,,,,,
2,video_forward_list,1656218,2,1647881.0,99.497,8337.0,0.503,,,,...,,,,,,,,,,
3,video_collect_list,1656218,2,1640545.0,99.054,15673.0,0.946,,,,...,,,,,,,,,,
4,video_neg_feedback_list,1656218,1,1656218.0,100.0,,,,,,...,,,,,,,,,,
5,video_play_done_list,1656218,2,765646.0,46.229,890572.0,53.771,,,,...,,,,,,,,,,
6,video_history_like_list,409416641,2,396789998.0,96.916,12626643.0,3.084,,,,...,,,,,,,,,,
7,video_history_comment_list,409416641,2,404849794.0,98.885,4566847.0,1.115,,,,...,,,,,,,,,,
8,video_history_forward_list,409416641,2,407776285.0,99.599,1640356.0,0.401,,,,...,,,,,,,,,,
9,video_history_collect_list,409416641,2,406667086.0,99.328,2749555.0,0.672,,,,...,,,,,,,,,,


## §2.8.2 Count 型 Label 分位数（决定「多次浏览」阈值）

对 `live_hist_valid_play_cnt` 等 count 类，看**非零行**的 P50/P90/P99：
- 如果 P50 = 1、P90 = 3，说明大部分「非零」都是「浏览过 1~3 次」→ 阈值定义可以是「>= 3 = 深度浏览」。
- 如果 P90 << P99，说明有极端重度用户，需要 clip。

In [4]:
cnt_rows = []
for f, arr in stats['count_pool'].items():
    if arr.size == 0: continue
    nz = arr[arr > 0]
    cnt_rows.append({
        'field':      f,
        'sample_n':   int(arr.size),
        'zero_%':     round(float((arr == 0).mean() * 100), 2),
        'nz_p50':     int(np.percentile(nz, 50)) if nz.size else 0,
        'nz_p90':     int(np.percentile(nz, 90)) if nz.size else 0,
        'nz_p99':     int(np.percentile(nz, 99)) if nz.size else 0,
        'nz_max':     int(nz.max()) if nz.size else 0,
        'nz_mean':    round(float(nz.mean()), 2) if nz.size else 0.0,
        'suggested_deep_thresh': int(np.percentile(nz, 90)) if nz.size else 0,  # 建议「深度」阈值
    })
cnt_df = pd.DataFrame(cnt_rows)
cnt_df.to_csv(OUT_DIR / 'count_label_percentiles.csv', index=False)
print('saved:', OUT_DIR / 'count_label_percentiles.csv')
cnt_df

saved: /Users/cjs/Desktop/MY/explore_llm_rec/analysis/outputs/labels/count_label_percentiles.csv


,field,sample_n,zero_%,nz_p50,nz_p90,nz_p99,nz_max,nz_mean,suggested_deep_thresh
0,live_hist_show_cnt_list,2000000,47.77,1,1,3,664,1.09,1
1,live_hist_play_cnt_list,2000000,56.88,1,5,13,460,2.29,5
2,live_hist_valid_play_cnt_list,2000000,68.51,1,3,10,460,1.86,3
3,live_hist_play_duration_list,2000000,56.88,27552,908093,4944733,84101426,344924.16,908093
4,live_hist_valid_play_duration_list,2000000,68.51,78243,1280936,5594443,84101426,470245.03,1280936
5,live_hist_like_cnt_list,2000000,94.87,14,234,1936,45064,127.17,234
6,live_hist_comment_cnt_list,2000000,93.56,4,35,405,88806,30.94,35
7,live_hist_reduce_similar_cnt_list,2000000,99.95,1,1,3,16,1.10,1
8,live_hist_report_live_cnt_list,2000000,99.98,1,4,10,17,2.04,4
9,live_hist_follow_author_cnt_list,2000000,89.63,1,1,1,103,1.02,1


## §2.8.3 字符串枚举字段分布（评估侧动作词的原始信号）

In [5]:
for f, dd in stats['str_dist'].items():
    if not dd: continue
    top = sorted(dd.items(), key=lambda x: -x[1])[:20]
    total = sum(dd.values())
    df = pd.DataFrame([{'value': v, 'count': c, 'pct': round(c/total*100, 2)} for v, c in top])
    safe = f.replace('/', '_').replace(' ', '_')
    df.to_csv(OUT_DIR / f'str_enum_dist_{safe}.csv', index=False)
    print(f'\n=== {f}   (distinct={len(dd)}, total={total:,}) ===')
    print(df.to_string(index=False))
print(f'\nsaved  str_enum_dist_*.csv')


=== live_hist_author_category_type_list   (distinct=5, total=1,000,000) ===
value  count   pct
    B 561940 56.19
    A 372145 37.21
 职业电商  63846  6.38
    C   1804  0.18
    D    265  0.03

=== live_hist_author_type_list   (distinct=4, total=1,000,000) ===
value  count   pct
 秀场主播 652294 65.23
   大V 151890 15.19
 游戏主播 142308 14.23
 电商主播  53508  5.35

=== outer_loop_history_action_pid_list_click_type   (distinct=10, total=1,000,000) ===
                               value  count   pct
                       AD_ITEM_CLICK 865978 86.60
                    EVENT_CONVERSION 120670 12.07
          EVENT_PRIVATE_MESSAGE_SENT   4108  0.41
              EVENT_KEY_INAPP_ACTION   3588  0.36
                           EVENT_PAY   2684  0.27
                  EVENT_NEXTDAY_STAY   1269  0.13
                   EVENT_FORM_SUBMIT   1059  0.11
EVENT_EFFECTIVE_CUSTOMER_ACQUISITION    339  0.03
                        LEADS_SUBMIT    278  0.03
              EVENT_WECHAT_CONNECTED     27  0.00

=== out

## §2.8.4 Live 直播类型布尔字段

In [6]:
flag_rows = []
for f, dd in stats['flag_dist'].items():
    total = sum(dd.values())
    if total == 0: continue
    n_pos = dd.get(1, 0)
    flag_rows.append({
        'field':          f,
        'total':          total,
        'n_positive':     n_pos,
        'pos_rate_%':     round(n_pos / total * 100, 3),
        'distinct_vals':  sorted(dd.keys()),
    })
flag_df = pd.DataFrame(flag_rows)
flag_df.to_csv(OUT_DIR / 'live_type_flags.csv', index=False)
print('saved:', OUT_DIR / 'live_type_flags.csv')
flag_df

saved: /Users/cjs/Desktop/MY/explore_llm_rec/analysis/outputs/labels/live_type_flags.csv


,field,total,n_positive,pos_rate_%,distinct_vals
0,live_hist_is_interactive_mp_live_list,35580635,0,0.000,[0]
1,live_hist_is_building_live_list,35580635,2609,0.007,"[0, 1]"
2,live_hist_is_local_life_live_list,35580635,0,0.000,[0]
3,live_hist_is_detect_game_live_list,35580635,6148615,17.281,"[0, 1]"
4,live_hist_is_recruit_live_list,35580635,1694,0.005,"[0, 1]"


## §2.8.5 ⭐ 动作词映射表（评估对齐核心）

评估示例里出现的中文动作词（如 `[视频-长播]`、`[商品-购买]`、`[直播-关注]`）需要从原始 label 反推。

映射规则基于以下语义：
- **video 域**：`play_done=1` → 长播；`like/collect/forward/comment=1` → 相应互动；
- **goods 域**：`is_click=1` → 点击；`is_cart=1` → 加购；`is_buy=1` → 购买；`ec_cvr_label>0` → 转化。
- **ad 域**：pos 序列的所有 pid → 转化；click 序列的所有 pid → 点击。
- **live 域**：`valid_play_cnt > deep_thresh`（用 §2.8.2 的 P90）→ 深度观看；`follow_author=1` → 关注；`like_cnt > 0` → 点赞。

输出 `action_phrase_mapping.csv`：直接给到 SFT prompt 生成脚本用于 timeline 转文本。

In [7]:
# 从 §2.8.2 结果里读 live count 的深度阈值
live_deep_thresh = {}
for _, r in cnt_df.iterrows():
    live_deep_thresh[r['field']] = int(r['nz_p90'])

action_map = [
    # domain, primary_seq, label_field, condition, action_phrase, priority
    # video/video: play_done 是主 label（60% 正样本），优先级最高
    ('video/video',  'video_history_sampled_pid_list', 'video_history_play_done_list',   '=1',       '[视频-长播]',    10),
    ('video/video',  'video_history_sampled_pid_list', 'video_history_like_list',        '=1',       '[视频-点赞]',    9),
    ('video/video',  'video_history_sampled_pid_list', 'video_history_collect_list',     '=1',       '[视频-收藏]',    8),
    ('video/video',  'video_history_sampled_pid_list', 'video_history_forward_list',     '=1',       '[视频-转发]',    7),
    ('video/video',  'video_history_sampled_pid_list', 'video_history_comment_list',     '=1',       '[视频-评论]',    6),
    ('video/video',  'video_history_sampled_pid_list', 'video_history_neg_feedback_list', '=1',      '[视频-负反馈]',  1),  # 全零，实际不会触发
    # 兜底：同 primary 但无任何互动（浏览过但没长播/点赞）
    ('video/video',  'video_history_sampled_pid_list', None,                             None,       '[视频-浏览]',    0),

    # goods: buy > cart > click 优先级
    ('goods',        'ec_colossus_rs_item_id_list',    'ec_colossus_rs_is_buy_list',     '=1',       '[商品-购买]',    10),
    ('goods',        'ec_colossus_rs_item_id_list',    'ec_colossus_rs_is_cart_list',    '=1',       '[商品-加购]',    9),
    ('goods',        'ec_colossus_rs_item_id_list',    'ec_colossus_rs_is_click_list',   '=1',       '[商品-点击]',    5),
    ('goods',        'ec_colossus_rs_item_id_list',    None,                             None,       '[商品-曝光]',    0),
    ('goods',        'ec_good_click_item_id_list_extend', None,                          None,       '[商品-点击]',    5),
    ('goods',        'ec_good_order_item_id_list_extend', None,                          None,       '[商品-购买]',    10),
    ('goods',        'ec_item_id_list',                'ec_cvr_label_list',              '>0',       '[商品-转化]',    10),

    # ad
    ('video/ad',     'outer_loop_history_action_pid_list_pos',    None,                  None,       '[广告-转化]',    10),
    ('video/ad',     'outer_loop_history_action_pid_list_click',  None,                  None,       '[广告-点击]',    5),
    ('video/ad',     'outer_loop_deep_target_pid',                None,                  None,       '[广告-深度转化]', 10),

    # live
    ('live',         'live_hist_author_id_list',       'live_hist_follow_author_cnt_list', '=1',      '[直播-关注]',   10),
    ('live',         'live_hist_author_id_list',       'live_hist_valid_play_cnt_list',   f'>={live_deep_thresh.get("live_hist_valid_play_cnt_list", 3)}', '[直播-深度观看]', 8),
    ('live',         'live_hist_author_id_list',       'live_hist_like_cnt_list',        '>0',       '[直播-点赞]',    7),
    ('live',         'live_hist_author_id_list',       'live_hist_comment_cnt_list',     '>0',       '[直播-评论]',    6),
    ('live',         'live_hist_author_id_list',       'live_hist_valid_play_cnt_list',   '>0',       '[直播-观看]',    3),
    ('live',         'live_hist_author_id_list',       'live_hist_reduce_similar_cnt_list', '>0',     '[直播-负反馈]',  1),
]

am_df = pd.DataFrame(action_map, columns=[
    'domain', 'primary_seq', 'label_field', 'condition', 'action_phrase', 'priority',
])
am_df.to_csv(OUT_DIR / 'action_phrase_mapping.csv', index=False)
print('saved:', OUT_DIR / 'action_phrase_mapping.csv')
print('\n=== 完整动作词表 ===')
am_df

saved: /Users/cjs/Desktop/MY/explore_llm_rec/analysis/outputs/labels/action_phrase_mapping.csv

=== 完整动作词表 ===


,domain,primary_seq,label_field,condition,action_phrase,priority
0,video/video,video_history_sampled_pid_list,video_history_play_done_list,=1,[视频-长播],10
1,video/video,video_history_sampled_pid_list,video_history_like_list,=1,[视频-点赞],9
2,video/video,video_history_sampled_pid_list,video_history_collect_list,=1,[视频-收藏],8
3,video/video,video_history_sampled_pid_list,video_history_forward_list,=1,[视频-转发],7
4,video/video,video_history_sampled_pid_list,video_history_comment_list,=1,[视频-评论],6
5,video/video,video_history_sampled_pid_list,video_history_neg_feedback_list,=1,[视频-负反馈],1
6,video/video,video_history_sampled_pid_list,NaN,NaN,[视频-浏览],0
7,goods,ec_colossus_rs_item_id_list,ec_colossus_rs_is_buy_list,=1,[商品-购买],10
8,goods,ec_colossus_rs_item_id_list,ec_colossus_rs_is_cart_list,=1,[商品-加购],9
9,goods,ec_colossus_rs_item_id_list,ec_colossus_rs_is_click_list,=1,[商品-点击],5


In [8]:
# ---- 用法 demo：给一条历史 pid 打上 action_phrase ----
def resolve_action_phrase(domain, primary_seq, label_snapshot):
    """给定 (domain, primary_seq_name, label_snapshot: dict of label_field->value) 返回优先级最高的动作词。
    label_snapshot 例子: {'video_history_play_done_list': 1, 'video_history_like_list': 0, ...}
    """
    candidates = am_df[(am_df.domain == domain) & (am_df.primary_seq == primary_seq)]
    for _, r in candidates.sort_values('priority', ascending=False).iterrows():
        if r['label_field'] is None or r['condition'] is None:
            return r['action_phrase']  # fallback 兜底
        v = label_snapshot.get(r['label_field'])
        if v is None: continue
        cond = r['condition']
        if cond == '=1' and v == 1: return r['action_phrase']
        if cond == '>0' and v > 0: return r['action_phrase']
        if cond.startswith('>=') and v >= int(cond[2:]): return r['action_phrase']
    return '[未知动作]'


# demo
print(resolve_action_phrase('video/video', 'video_history_sampled_pid_list',
    {'video_history_play_done_list': 1, 'video_history_like_list': 1}))  # → 长播（优先级 10 > 点赞 9）
print(resolve_action_phrase('video/video', 'video_history_sampled_pid_list',
    {'video_history_play_done_list': 0, 'video_history_like_list': 1}))  # → 点赞
print(resolve_action_phrase('video/video', 'video_history_sampled_pid_list',
    {'video_history_play_done_list': 0, 'video_history_like_list': 0}))  # → 浏览（fallback）
print(resolve_action_phrase('goods', 'ec_colossus_rs_item_id_list',
    {'ec_colossus_rs_is_buy_list': 1}))  # → 购买
print(resolve_action_phrase('live', 'live_hist_author_id_list',
    {'live_hist_follow_author_cnt_list': 1, 'live_hist_valid_play_cnt_list': 5}))  # → 关注

[视频-长播]
[视频-点赞]
[未知动作]
[商品-购买]
[直播-关注]


## §2.8.6 结论 & 对 SFT 的直接指导（抄进 report.md）

1. **数值 label 精细分布**：确认 `ec_cvr_label` 是 0/1（不是多值）；video 域 like/comment/collect/forward 都严格 0/1 —— 二值 label 用 BCE loss。
2. **count-type 阈值**：`live_hist_valid_play_cnt` 的 P90 就是「深度观看」阈值（大概率 3~5）；`watch_time_list` P90 是「长播时长」阈值。
3. **字符串枚举**：`click_industry` 有多少 distinct → 决定 SFT prompt 里是把 industry 当自由文本还是特殊 token。
4. **Live 类型标签**：5 个直播类型 flag 的正样本率都不高（<10%），可作为**副 label**（多任务 loss 的一小部分）而非主 label。
5. **⭐ 动作词映射**：`action_phrase_mapping.csv` 直接给 SFT 样本构造脚本用。 timeline 里每一条历史 pid 都要根据 label 快照 + 优先级规则打上一个动作词。

### 与 §2.5 建议 U 的一致性

§2.5 建议 U 里已经给出了正样本定义 & focal loss 权重，本 notebook 的动作词映射与之**完全一致**（`play_done` 是 video 主动作 / `is_click` 是 goods 主动作 / `follow` 是 live 主动作），只是把「二元正负样本」升级为「多类中文动作词」，用于 SFT prompt 的 timeline 描述。

### 产出物清单（`analysis/outputs/labels/`）
- `numeric_label_dist.csv` —— 数值 label 精确分布
- `count_label_percentiles.csv` —— count 型 P50/P90/P99 & 深度阈值
- `str_enum_dist_{field}.csv` × 4
- `live_type_flags.csv`
- **`action_phrase_mapping.csv` ⭐** —— 动作词映射表
- `_cache.pkl`